# Gas Prices — Feature Engineering

Builds an enriched dataset by joining gas price data with external market signals and engineering temporal event features.

**Output:** `data/processed/gas_prices_enriched.csv` — 132 columns, 1409 weekly rows, 2000–2026

**Features added:**
- External signals: WTI crude, natural gas, gold, VIX, USD/CAD
- Provincial carbon tax rates (BC, ON, AB)
- Temporal event windows: T-12w, T-8w, T-4w, T-2w before each event
- Event classification: type and oil proximity
- National median and provincial medians
- Price momentum: rolling 2w, 4w, 8w, 12w changes
- Provincial deviation from national median

## Cell 1 — Imports and paths

In [1]:
import pandas as pd
import numpy as np
import subprocess
from pathlib import Path

subprocess.run(['pip', 'install', 'yfinance', '--break-system-packages', '-q'])
import yfinance as yf

BASE_DIR         = Path().resolve()
processed_folder = BASE_DIR / 'data' / 'processed'
figures_folder   = BASE_DIR / 'figures'
figures_folder.mkdir(exist_ok=True)

print(f'Processed : {processed_folder}')

Processed : /Users/btosi/projects/gas_price/data/processed


## Cell 2 — Load cleaned gas prices

In [2]:
df = pd.read_csv(
    processed_folder / 'gas_prices_all_years.csv',
    index_col='date',
    parse_dates=True
)

print(f'Shape     : {df.shape}')
print(f'Date range: {df.index.min().date()} → {df.index.max().date()}')

Shape     : (1409, 82)
Date range: 2000-01-04 → 2026-12-29


## Cell 3 — Event definitions

All global events with type and oil proximity classification.
Derived from EDA findings — oil proximity is the key differentiator within supply_conflict events.

In [3]:
EVENTS = {
    '9/11 attacks'          : '2001-09-11',
    'Iraq War'              : '2003-03-20',
    'Hurricane Katrina'     : '2005-08-29',
    '2008 financial crisis' : '2008-09-15',
    'Arab Spring / Libya'   : '2011-02-17',
    'Crimea annexation'     : '2014-03-18',
    'OPEC production cut'   : '2016-11-30',
    'US-Iran / Soleimani'   : '2020-01-03',
    'COVID-19'              : '2020-03-15',
    'Ukraine invasion'      : '2022-02-24',
    'Gaza conflict'         : '2023-10-07',
}

EVENT_TYPES = {
    '9/11 attacks'          : 'demand_shock',
    'Iraq War'              : 'supply_conflict',
    'Hurricane Katrina'     : 'supply_disruption',
    '2008 financial crisis' : 'demand_shock',
    'Arab Spring / Libya'   : 'supply_conflict',
    'Crimea annexation'     : 'supply_conflict',
    'OPEC production cut'   : 'policy',
    'US-Iran / Soleimani'   : 'supply_conflict',
    'COVID-19'              : 'demand_shock',
    'Ukraine invasion'      : 'supply_conflict',
    'Gaza conflict'         : 'supply_conflict',
}

OIL_PROXIMITY = {
    '9/11 attacks'          : 'indirect',
    'Iraq War'              : 'direct',
    'Hurricane Katrina'     : 'direct',
    '2008 financial crisis' : 'indirect',
    'Arab Spring / Libya'   : 'direct',
    'Crimea annexation'     : 'indirect',
    'OPEC production cut'   : 'direct',
    'US-Iran / Soleimani'   : 'moderate',
    'COVID-19'              : 'indirect',
    'Ukraine invasion'      : 'direct',
    'Gaza conflict'         : 'indirect',
}

event_dates = [pd.Timestamp(d) for d in EVENTS.values()]
print('✅ Events defined:', list(EVENTS.keys()))

✅ Events defined: ['9/11 attacks', 'Iraq War', 'Hurricane Katrina', '2008 financial crisis', 'Arab Spring / Libya', 'Crimea annexation', 'OPEC production cut', 'US-Iran / Soleimani', 'COVID-19', 'Ukraine invasion', 'Gaza conflict']


## Cell 4 — Pull external market signals via Yahoo Finance

Coverage notes (Yahoo Finance limitations):
- WTI, natgas, gold: Aug 2000 → present (8 months NaN at start)
- VIX: Jan 2000 → present (full coverage)
- USD/CAD: Sep 2003 → present
- Brent excluded — high WTI correlation, worse coverage (starts 2007)

In [4]:
tickers = {
    'WTI_crude_usd' : 'CL=F',
    'natgas_usd'    : 'NG=F',
    'gold_usd'      : 'GC=F',
    'VIX'           : '^VIX',
    'USD_CAD'       : 'CAD=X',
}

external = {}
for name, ticker in tickers.items():
    try:
        data = yf.download(ticker, start='2000-01-01', interval='1wk', progress=False)
        s = data[['Close']].copy()
        s.columns = [name]
        s.index = pd.to_datetime(s.index).tz_localize(None)
        external[name] = s
        print(f'✅ {name}: {s.index.min().date()} → {s.index.max().date()}, rows: {len(s)}')
    except Exception as e:
        print(f'❌ {name}: {e}')

# Combine and resample to weekly Tuesday
ext_df = pd.concat(external.values(), axis=1, sort=True)
ext_df.index = pd.to_datetime(ext_df.index)
ext_df.index.name = 'date'
ext_df = ext_df.resample('W-TUE').mean()

# Forward fill small gaps only (market closures) — max 2 weeks
small_gap_cols = ['WTI_crude_usd', 'natgas_usd', 'gold_usd', 'VIX']
ext_df[small_gap_cols] = ext_df[small_gap_cols].ffill(limit=2)

print(f'\nExternal signals shape: {ext_df.shape}')
print('NaN counts (genuine missing history):')
print(ext_df.isna().sum())

✅ WTI_crude_usd: 2000-08-21 → 2026-04-06, rows: 1338
✅ natgas_usd: 2000-08-28 → 2026-04-06, rows: 1337
✅ gold_usd: 2000-08-28 → 2026-04-06, rows: 1337
✅ VIX: 2000-01-01 → 2026-04-04, rows: 1371
✅ USD_CAD: 2003-09-15 → 2026-04-06, rows: 1178

External signals shape: (1371, 5)
NaN counts (genuine missing history):
WTI_crude_usd     33
natgas_usd        34
gold_usd          34
VIX                0
USD_CAD          193
dtype: int64


## Cell 5 — Provincial carbon tax rates

Built manually from public policy records (Environment Canada, provincial announcements).
Units: CAD cents per litre of gasoline.

In [5]:
carbon_tax_events = [
    ('BC', '2008-07-01',  2.41), ('BC', '2009-07-01',  4.82),
    ('BC', '2010-07-01',  7.24), ('BC', '2011-07-01',  9.65),
    ('BC', '2012-07-01', 12.06), ('BC', '2013-07-01', 13.27),
    ('BC', '2018-04-01', 14.43), ('BC', '2019-04-01', 17.61),
    ('BC', '2020-04-01', 22.35), ('BC', '2021-04-01', 26.46),
    ('BC', '2022-04-01', 30.52), ('BC', '2023-04-01', 34.52),
    ('BC', '2024-04-01', 38.57),
    ('ON', '2019-04-01',  4.42), ('ON', '2020-04-01',  6.63),
    ('ON', '2021-04-01',  8.84), ('ON', '2022-04-01', 11.05),
    ('ON', '2023-04-01', 14.31), ('ON', '2024-04-01', 17.61),
    ('AB', '2007-01-01',  0.00), ('AB', '2020-01-01',  6.63),
    ('AB', '2021-04-01',  8.84), ('AB', '2022-04-01', 11.05),
    ('AB', '2023-04-01', 14.31), ('AB', '2024-04-01', 17.61),
]

date_range = pd.date_range('2000-01-01', '2026-12-31', freq='W-TUE')
carbon_df  = pd.DataFrame(index=date_range)
carbon_df.index.name = 'date'

for prov in ['BC', 'ON', 'AB']:
    events = sorted([(pd.Timestamp(d), r) for p, d, r in carbon_tax_events if p == prov])
    rates  = pd.Series(0.0, index=date_range)
    for date, rate in events:
        rates.loc[date:] = rate
    carbon_df[f'{prov}_carbon_tax'] = rates

print('✅ Carbon tax series built')
print(carbon_df.tail(3))

✅ Carbon tax series built
            BC_carbon_tax  ON_carbon_tax  AB_carbon_tax
date                                                   
2026-12-15          38.57          17.61          17.61
2026-12-22          38.57          17.61          17.61
2026-12-29          38.57          17.61          17.61


## Cell 6 — Join all data sources

In [6]:
df_enriched = df.join(ext_df, how='left').join(carbon_df, how='left')

print(f'Shape: {df_enriched.shape}')
print(f'Date range: {df_enriched.index.min().date()} → {df_enriched.index.max().date()}')

Shape: (1409, 90)
Date range: 2000-01-04 → 2026-12-29


## Cell 7 — Temporal event window features

For each row, compute:
- Days to nearest event (any direction)
- Days to next upcoming event
- Binary flags: is this row within T-12w, T-8w, T-4w, T-2w of the next event?
- Event type and oil proximity of the next upcoming event
- Key binary: is this a direct oil supply conflict in the pre-event window?

In [7]:
df_enriched['days_to_nearest_event'] = df_enriched.index.map(
    lambda d: min([abs((d - e).days) for e in event_dates])
)

def days_to_next_event(date):
    future = [e for e in event_dates if e >= date]
    return (min(future) - date).days if future else np.nan

def nearest_event_attr(date, attr_dict):
    future = [(e, name) for name, e in zip(EVENTS.keys(), event_dates) if e >= date]
    if not future:
        return 'none'
    return attr_dict[min(future, key=lambda x: x[0])[1]]

df_enriched['days_to_next_event']    = df_enriched.index.map(days_to_next_event)
df_enriched['is_12w_before_event']   = (df_enriched['days_to_next_event'] <= 84).astype(int)
df_enriched['is_8w_before_event']    = (df_enriched['days_to_next_event'] <= 56).astype(int)
df_enriched['is_4w_before_event']    = (df_enriched['days_to_next_event'] <= 28).astype(int)
df_enriched['is_2w_before_event']    = (df_enriched['days_to_next_event'] <= 14).astype(int)
df_enriched['next_event_type']       = df_enriched.index.map(lambda d: nearest_event_attr(d, EVENT_TYPES))
df_enriched['next_event_proximity']  = df_enriched.index.map(lambda d: nearest_event_attr(d, OIL_PROXIMITY))

df_enriched['is_direct_oil_conflict'] = (
    (df_enriched['next_event_type']      == 'supply_conflict') &
    (df_enriched['next_event_proximity'] == 'direct') &
    (df_enriched['is_8w_before_event']   == 1)
).astype(int)

print('✅ Event features added')

✅ Event features added


## Cell 8 — Provincial medians, national median, momentum, deviations

In [8]:
PROVINCES = {
    'BC'  : ['VANCOUVER', 'VICTORIA', 'PRINCE GEORGE', 'KAMLOOPS', 'KELOWNA', 'ABBOTSFORD', 'FORT ST. JOHN'],
    'AB'  : ['CALGARY', 'EDMONTON', 'RED DEER', 'LETHBRIDGE', 'GRANDE PRAIRIE', 'LLOYDMINSTER'],
    'SK'  : ['REGINA', 'SASKATOON', 'PRINCE ALBERT', 'MOOSE JAW'],
    'MB'  : ['WINNIPEG', 'BRANDON'],
    'ON'  : ['CITY OF TORONTO', 'OTTAWA', 'WINDSOR', 'LONDON', 'SUDBURY', 'SAULT STE MARIE',
             'THUNDER BAY', 'NORTH BAY', 'TIMMINS', 'HAMILTON', 'ST. CATHARINES',
             'BARRIE', 'BRANTFORD', 'GUELPH', 'KITCHENER', 'KINGSTON',
             'OSHAWA', 'PETERBOROUGH', 'PEEL REGION'],
    'QC'  : ['MONTRÉAL', 'QUÉBEC', 'SHERBROOKE', 'GASPÉ', 'CHICOUTIMI', 'RIMOUSKI',
             'TROIS RIVIÈRES', 'DRUMMONDVILLE', "VAL D'OR", 'GATINEAU'],
    'NB'  : ['SAINT JOHN', 'FREDERICTON', 'MONCTON', 'BATHURST', 'EDMUNDSTON',
             'MIRAMICHI', 'CAMPBELLTON', 'SUSSEX', 'WOODSTOCK'],
    'NS'  : ['HALIFAX', 'SYDNEY', 'YARMOUTH', 'TRURO', 'KENTVILLE', 'NEW GLASGOW'],
    'PEI' : ['CHARLOTTETOWN'],
    'NL'  : ['ST JOHNS', 'GANDER', 'CORNER BROOK', 'GRAND FALLS', 'LABRADOR CITY'],
    'YT'  : ['WHITEHORSE'],
    'NT'  : ['YELLOWKNIFE'],
}

# Build all new columns separately then join at once — avoids fragmentation
new_cols = {}

# Provincial medians
prov_median_cols = []
for prov, cities in PROVINCES.items():
    available = [c for c in cities if c in df_enriched.columns]
    col = f'{prov}_median'
    new_cols[col] = df_enriched[available].median(axis=1)
    prov_median_cols.append(col)

prov_medians_df = pd.DataFrame(new_cols, index=df_enriched.index)

# National median
national_median = prov_medians_df.median(axis=1)

# Price momentum
momentum = {}
for w in [2, 4, 8, 12]:
    momentum[f'national_chg_{w}w'] = national_median.diff(w)
    momentum[f'national_pct_{w}w'] = national_median.pct_change(w) * 100
momentum_df = pd.DataFrame(momentum, index=df_enriched.index)

# Provincial deviations
deviations = {f'{p}_dev': prov_medians_df[f'{p}_median'] - national_median for p in PROVINCES}
deviations_df = pd.DataFrame(deviations, index=df_enriched.index)

# Join all at once
df_enriched = pd.concat([
    df_enriched,
    prov_medians_df,
    national_median.rename('national_median'),
    momentum_df,
    deviations_df
], axis=1)

# Defragment
df_enriched = df_enriched.copy()

print(f'✅ Province and momentum features added')
print(f'Shape: {df_enriched.shape}')

✅ Province and momentum features added
Shape: (1409, 132)


## Cell 9 — Save enriched dataset

In [9]:
output_path = processed_folder / 'gas_prices_enriched.csv'
df_enriched.to_csv(output_path)

print(f'✅ Saved to {output_path}')
print(f'   Shape     : {df_enriched.shape}')
print(f'   Date range: {df_enriched.index.min().date()} → {df_enriched.index.max().date()}')
print(f'   Columns   : {df_enriched.shape[1]}')
print(f'\nNaN summary (>0% only):')
nan_summary = (df_enriched.isna().mean() * 100).round(1)
print(nan_summary[nan_summary > 0].to_string())

✅ Saved to /Users/btosi/projects/gas_price/data/processed/gas_prices_enriched.csv
   Shape     : (1409, 132)
   Date range: 2000-01-04 → 2026-12-29
   Columns   : 132

NaN summary (>0% only):
ABBOTSFORD              60.0
Atlantic Ave(P)          2.8
BARRIE                  60.0
BATHURST                 2.8
BRANDON                  2.8
BRANTFORD               60.0
CALGARY                  2.8
CAMPBELLTON             25.9
CHARLOTTETOWN            2.8
CHICOUTIMI               2.8
CITY OF TORONTO          2.8
CORNER BROOK             2.8
Canada Ave(V)            2.8
Canada(P)                2.8
Canada(S)                2.8
DRUMMONDVILLE           25.9
EDMONTON                 2.8
EDMUNDSTON              25.0
FORT ST. JOHN           25.9
FREDERICTON              2.8
GANDER                   2.8
GASPÉ                    2.8
GATINEAU                63.9
GRAND FALLS             63.9
GRANDE PRAIRIE          63.9
GUELPH                  60.0
HALIFAX                  2.8
HAMILTON                 